# 08 - Logs de Auditoria
## Registro de Ejecucion y Metricas del Pipeline

---

### Objetivo de este notebook

Este notebook recopila **metricas de auditoria** de todas las tablas del pipeline
y las almacena en una tabla historica de log para trazabilidad.

### Por que registrar logs de auditoria?

En un pipeline de produccion, los logs de auditoria son fundamentales para:

- **Trazabilidad**: Saber cuando se ejecuto el pipeline y cuantos registros tenia cada tabla
- **Deteccion de anomalias**: Comparar metricas entre ejecuciones para detectar cambios inesperados
- **Cumplimiento**: Demostrar que los datos se procesan correctamente (compliance)
- **Depuracion**: Facilitar la investigacion cuando algo sale mal

### Tabla de auditoria

Los registros se almacenan en: `pubchem_oro.log_ejecuciones`

| Columna | Tipo | Descripcion |
|---------|------|-------------|
| id_ejecucion | STRING | Identificador unico de la ejecucion (timestamp) |
| fecha_ejecucion | TIMESTAMP | Fecha y hora exacta |
| capa | STRING | Capa del pipeline (raw, bronce, plata, oro) |
| tabla | STRING | Nombre completo de la tabla auditada |
| num_registros | LONG | Cantidad de registros en la tabla |
| num_columnas | INT | Cantidad de columnas |
| estado | STRING | disponible / no_disponible |
| observacion | STRING | Observaciones adicionales |

---
## 1. Importacion de Librerias y Configuracion

In [ ]:
from pyspark.sql import functions as F  # Funciones de transformacion de Spark SQL
from pyspark.sql.types import (          # Tipos de datos para definir el esquema del log
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)
from datetime import datetime            # Para generar marcas de tiempo

In [ ]:
# Configuracion de catalogo y tabla de destino
CATALOGO = spark.sql("SELECT current_catalog()").collect()[0][0]
ESQUEMA_ORO = "pubchem_oro"                        # Esquema donde reside la tabla de log
TABLA_LOG = f"{ESQUEMA_ORO}.log_ejecuciones"       # Nombre completo de la tabla de auditoria

print(f"Catalogo: {CATALOGO}")
print(f"Tabla de auditoria: {TABLA_LOG}")

---
## 2. Crear Tabla de Auditoria

Se crea la tabla de auditoria si no existe previamente.
La sentencia `CREATE TABLE IF NOT EXISTS` es idempotente:
si la tabla ya existe, no se modifica.

Se utiliza formato Delta para aprovechar las capacidades de
versionado (time travel) y ACID del formato.

In [ ]:
# Eliminar la tabla si existe para recrearla con el esquema correcto
spark.sql(f"DROP TABLE IF EXISTS {TABLA_LOG}")

# Crear la tabla de log de auditoria con esquema explicito
# Se usa LONG en lugar de INT para num_columnas para evitar conflictos de tipos
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLA_LOG} (
        id_ejecucion STRING COMMENT 'Identificador unico de la ejecucion',
        fecha_ejecucion TIMESTAMP COMMENT 'Fecha y hora de la ejecucion',
        capa STRING COMMENT 'Capa del pipeline (raw, bronce, plata, oro)',
        tabla STRING COMMENT 'Nombre completo de la tabla',
        num_registros LONG COMMENT 'Cantidad de registros en la tabla',
        num_columnas LONG COMMENT 'Cantidad de columnas en la tabla',
        estado STRING COMMENT 'Estado de la tabla (disponible, no_disponible)',
        observacion STRING COMMENT 'Observaciones adicionales'
    )
    USING DELTA
    COMMENT 'Tabla de auditoria para registrar metricas del pipeline en cada ejecucion'
""")

print(f"Tabla {TABLA_LOG} verificada/creada exitosamente.")

---
## 3. Recopilar Metricas de Todas las Tablas

Se itera sobre todas las tablas del pipeline (de todas las capas)
y se recopilan sus metricas: cantidad de registros, columnas y estado.

Para cada tabla se maneja el caso en que no este disponible
(por ejemplo, si el pipeline no se ejecuto completo).

In [ ]:
# Generar identificador unico para esta ejecucion
# El formato YYYYMMDD_HHMMSS garantiza unicidad y orden cronologico
fecha_actual = datetime.now()
id_ejecucion = fecha_actual.strftime("%Y%m%d_%H%M%S")

print(f"ID de ejecucion: {id_ejecucion}")
print(f"Fecha: {fecha_actual.strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Definir las tablas a auditar, organizadas por capa
# Cada tupla contiene: (nombre_capa, nombre_completo_tabla)
tablas_a_auditar = [
    ("bronce", "pubchem_bronce.propiedades_compuestos"),
    ("bronce", "pubchem_bronce.bioactividad"),
    ("plata",  "pubchem_plata.actividades_biologicas"),
    ("oro",    "pubchem_oro.dim_compuestos"),
    ("oro",    "pubchem_oro.dim_ensayos"),
    ("oro",    "pubchem_oro.dim_resultados"),
    ("oro",    "pubchem_oro.fact_bioactividades"),
]

# Lista para acumular las metricas como tuplas
metricas = []

# Iterar sobre cada tabla y recopilar sus metricas
for capa, nombre_tabla in tablas_a_auditar:
    try:
        # Intentar leer la tabla y obtener metricas
        df_tabla = spark.table(nombre_tabla)
        num_registros = df_tabla.count()       # Cantidad de filas
        num_columnas = len(df_tabla.columns)   # Cantidad de columnas
        estado = "disponible"
        observacion = "Tabla leida exitosamente"
        print(f"  [{capa.upper():7s}] {nombre_tabla:45s} -> {num_registros:>10,} registros, {num_columnas} columnas")

    except Exception as e:
        # Si la tabla no existe o no es accesible, registrar error
        num_registros = 0
        num_columnas = 0
        estado = "no_disponible"
        observacion = f"Error: {str(e)[:200]}"
        print(f"  [{capa.upper():7s}] {nombre_tabla:45s} -> NO DISPONIBLE")

    # Agregar metrica como tupla a la lista
    metricas.append((
        id_ejecucion,
        fecha_actual,
        capa,
        nombre_tabla,
        num_registros,
        num_columnas,
        estado,
        observacion
    ))

---
## 4. Auditar Capa Raw (Archivos en Volumen)

La capa Raw no tiene tablas Delta sino archivos JSON en un volumen.
Se audita de forma diferente: contando archivos y su tamano total.

In [ ]:
# Ruta del volumen de archivos crudos
RUTA_RAW = f"/Volumes/{CATALOGO}/pubchem_raw/bioactividad"

try:
    # Listar archivos en el volumen y calcular metricas
    archivos_raw = dbutils.fs.ls(RUTA_RAW)
    num_archivos = len(archivos_raw)                          # Cantidad de archivos
    tamano_total = sum(archivo.size for archivo in archivos_raw)  # Tamano total en bytes
    observacion_raw = f"{num_archivos} archivos, {tamano_total:,} bytes totales"
    estado_raw = "disponible"
    print(f"  [RAW    ] {RUTA_RAW} -> {observacion_raw}")

except Exception as e:
    # El volumen no existe o no es accesible
    num_archivos = 0
    estado_raw = "no_disponible"
    observacion_raw = f"Error: {str(e)[:200]}"
    print(f"  [RAW    ] {RUTA_RAW} -> NO DISPONIBLE")

# Agregar metrica de la capa Raw a la lista
metricas.append((
    id_ejecucion,
    fecha_actual,
    "raw",
    f"pubchem_raw.bioactividad (volumen)",
    num_archivos,      # Para Raw, num_registros = num_archivos
    0,                 # No aplica num_columnas para archivos
    estado_raw,
    observacion_raw
))

---
## 5. Insertar Registros de Auditoria

Se crea un DataFrame de Spark con las metricas recopiladas y se insertan
en la tabla de auditoria con modo **append** para preservar el historial
de todas las ejecuciones anteriores.

In [ ]:
# Definir el esquema del DataFrame de metricas
# Debe coincidir con la estructura de la tabla de auditoria
esquema_log = StructType([
    StructField("id_ejecucion", StringType(), False),      # No nullable
    StructField("fecha_ejecucion", TimestampType(), False), # No nullable
    StructField("capa", StringType(), False),               # No nullable
    StructField("tabla", StringType(), False),              # No nullable
    StructField("num_registros", LongType(), True),         # Nullable
    StructField("num_columnas", LongType(), True),          # Nullable
    StructField("estado", StringType(), True),              # Nullable
    StructField("observacion", StringType(), True),         # Nullable
])

# Crear DataFrame con las metricas recopiladas
df_metricas = spark.createDataFrame(metricas, schema=esquema_log)

# Insertar con modo append para preservar historial de ejecuciones anteriores
# A diferencia de overwrite, append agrega nuevos registros sin borrar los existentes
df_metricas.write.format("delta") \
    .mode("append") \
    .saveAsTable(TABLA_LOG)

# Confirmar la insercion
registros_insertados = df_metricas.count()
print(f"\nRegistros de auditoria insertados: {registros_insertados}")
print(f"ID de ejecucion: {id_ejecucion}")

---
## 6. Verificacion del Log de Auditoria

Se verifican los registros insertados en esta ejecucion y se muestra
el historial de las ultimas ejecuciones registradas.

In [ ]:
# Mostrar los registros de la ejecucion actual filtrados por id_ejecucion
print(f"--- Registros de auditoria para ejecucion {id_ejecucion} ---")
spark.table(TABLA_LOG).filter(
    F.col("id_ejecucion") == id_ejecucion
).orderBy("capa", "tabla").show(truncate=False)

In [ ]:
# Mostrar historial de las ultimas 5 ejecuciones
# Esto permite ver la frecuencia de ejecucion del pipeline
print("--- Historial de ejecuciones recientes ---")
spark.table(TABLA_LOG).select(
    "id_ejecucion", "fecha_ejecucion"
).distinct().orderBy(
    F.desc("fecha_ejecucion")
).limit(5).show(truncate=False)

# Mostrar totales historicos del log
total_log = spark.table(TABLA_LOG).count()
ejecuciones_unicas = spark.table(TABLA_LOG).select("id_ejecucion").distinct().count()

print(f"Total de registros en log: {total_log}")
print(f"Ejecuciones registradas:   {ejecuciones_unicas}")